# Exercise — Trace Lineage & Specify Monitoring

**Trailhead Provisions** is pre-instrumented with OpenLineage-shaped pipeline lineage. Use it to
answer two separate governance questions:
1. **Customer PII breach blast radius (downstream):** If `customer` is breached, which downstream
   datasets are exposed (and does it reach the CSRD carbon report)?
2. **CSRD report provenance (upstream):** Which upstream tables actually feed
   `quarterly_sustainability`?

Then specify the monitoring table that protects both paths. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog and S3 tables are seeded in your lab account.

In [1]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

display(gc.lineage.events())
print("Datasets:", sorted(gc.lineage.datasets()))

Backend: live -> AwsGovernedCatalog (Glue DB: trailhead, Region: us-east-1)


,job,inputs,outputs
0,build_order_carbon,"orders, shipment",order_carbon
1,build_quarterly_sustainability,order_carbon,quarterly_sustainability
2,build_customer_360,"customer, campaign_membership",customer_360
3,build_marketing_audience,"customer_360, orders",marketing_audience


Datasets: ['campaign_membership', 'customer', 'customer_360', 'marketing_audience', 'order_carbon', 'orders', 'quarterly_sustainability', 'shipment']


## 1. Trace impact (two separate lineage questions)
1. **Question A — PII breach blast radius (downstream):** Trace everything **downstream of `customer`** (`gc.lineage.trace_downstream("customer")`) to confirm which datasets a customer PII breach impacts (and verify that it **does not** reach `quarterly_sustainability`).
2. **Question B — CSRD report provenance (upstream):** Trace the **upstream sources** of `quarterly_sustainability` (`gc.lineage.trace_upstream("quarterly_sustainability")`) to confirm the CSRD report's actual upstream sources.

In [3]:
# 1. Question A: Downstream blast radius of a customer PII breach
pii_impact = gc.lineage.trace_downstream("customer")
print("1. Downstream of a customer PII breach:", pii_impact)

# 2. Question B: Upstream provenance of the CSRD quarterly sustainability report
report_sources = gc.lineage.trace_upstream("quarterly_sustainability")
print("2. Upstream sources of the CSRD report:", report_sources)

1. Downstream of a customer PII breach: ['customer_360', 'marketing_audience']
2. Upstream sources of the CSRD report: ['order_carbon', 'orders', 'shipment']


## 2. Monitoring specification
Replace the cell below with your spec.

### Monitoring specification

| Signal | Where (Source dataset / job / log) | Threshold | Owner | SLA / Regulation |
|---|---|---|---|---|
| Anomalous PII read volume / unauthorized access on `customer` or `customer_360` | Lake Formation / CloudTrail data-access logs | `> 3x` 7-day rolling read baseline or any query by a principal without an LF grant | Security & Central Governance Lead (with Customer Identity Lead) | GDPR Art. 33 (72-hour breach notification window) |
| Carbon unit out of range (`carbon_kg`) | `shipment` → `build_order_carbon` | any `carbon_kg > 100` (0 rows above max) | Inventory/Fulfillment Lead | Accuracy SLO; CSRD Art. 29a report integrity |
| Consent mismatch rate | `build_customer_360` (`customer` ⋈ `campaign_membership`) | `> 1%` loyalty ≠ marketing consent | Customer Identity Lead (with Marketing Lead) | Consistency SLO; GDPR Art. 6 / Art. 7 consent |
| Freshness of `quarterly_sustainability` | `build_quarterly_sustainability` job | `> 24h` since last successful run | CSRD Reporting Owner | Report freshness SLA; CSRD filing schedule |
| Null mandatory attributes (`product.category`, `product.sustainability_class`) | `product` producer ingestion check | `> 1%` null rate (`>= 99.0%` non-null SLO) | Inventory/Fulfillment Lead | Completeness SLO; CSRD product classification |
| Lineage event missing | All scheduled ETL jobs (OpenLineage emitter) | Any job run completing with no emitted lineage event | Platform Engineering | Governance audit / observability |

**Lineage findings (two distinct subgraphs):**
1. **Customer PII breach blast radius (downstream of `customer`):** `['customer_360', 'marketing_audience']`. A breach or corruption of `customer` contaminates Customer 360 and Marketing Audience, requiring GDPR Art. 33 breach assessment within 72 hours, and **does not** reach `quarterly_sustainability`.
2. **CSRD report provenance (upstream of `quarterly_sustainability`):** `['order_carbon', 'orders', 'shipment']`. A unit defect in `shipment.carbon_kg` flows through `order_carbon` into `quarterly_sustainability` with no intervening quality gate.

**Gap the current signals miss:** There is no monitor on the `customer_id` ↔ `cust_ref` identity-resolution match rate in `build_customer_360` (nor on the ~6% unresolvable `cust_ref = 'n/a'` rows in `campaign_membership`), and `product` / `supplier` ESG attributes are completely disconnected from the sustainability lineage (`build_order_carbon` only ingests `orders` and `shipment`). Adding a match-rate monitor on `build_customer_360` and wiring `product`/`supplier` into the CSRD pipeline closes both gaps.

---

# Solution — Trace Lineage & Specify Monitoring

## Summary

The skill is reading lineage to answer **two separate operational questions** — (1) downstream
blast radius of a PII breach and (2) upstream provenance of a regulated report — and turning both
into a concrete monitoring specification with thresholds, owners, and regulatory SLAs.

## Key traces

```python
pii_impact = gc.lineage.trace_downstream("customer")
# -> ['customer_360', 'marketing_audience']

report_sources = gc.lineage.trace_upstream("quarterly_sustainability")
# -> ['order_carbon', 'orders', 'shipment']
```

1. **Downstream of `customer` (`['customer_360', 'marketing_audience']`):** Confirms that a
   `customer` PII breach impacts Customer 360 and Marketing Audience, and **does not** reach
   `quarterly_sustainability`.
2. **Upstream of `quarterly_sustainability` (`['order_carbon', 'orders', 'shipment']`):** Confirms
   the CSRD report's actual upstream sources (`order_carbon`, fed by `orders` and `shipment`), with
   no dependency on `customer`.

### Monitoring specification

| Signal | Where (Source dataset / job / log) | Threshold | Owner | SLA / Regulation |
|---|---|---|---|---|
| Anomalous PII read volume / unauthorized access on `customer` or `customer_360` | Lake Formation / CloudTrail data-access logs | `> 3x` 7-day rolling read baseline or any query by a principal without an LF grant | Security & Central Governance Lead (with Customer Identity Lead) | GDPR Art. 33 (72-hour breach notification window) |
| Carbon unit out of range (`carbon_kg`) | `shipment` → `build_order_carbon` | any `carbon_kg > 100` (0 rows above max) | Inventory/Fulfillment Lead | Accuracy SLO; CSRD Art. 29a report integrity |
| Consent mismatch rate | `build_customer_360` (`customer` ⋈ `campaign_membership`) | `> 1%` loyalty ≠ marketing consent | Customer Identity Lead (with Marketing Lead) | Consistency SLO; GDPR Art. 6 / Art. 7 consent |
| Freshness of `quarterly_sustainability` | `build_quarterly_sustainability` job | `> 24h` since last successful run | CSRD Reporting Owner | Report freshness SLA; CSRD filing schedule |
| Null mandatory attributes (`product.category`, `product.sustainability_class`) | `product` producer ingestion check | `> 1%` null rate (`>= 99.0%` non-null SLO) | Inventory/Fulfillment Lead | Completeness SLO; CSRD product classification |
| Lineage event missing | All scheduled ETL jobs (OpenLineage emitter) | Any job run completing with no emitted lineage event | Platform Engineering | Governance audit / observability |

**Gap the current signals miss:** Neither `product` nor `supplier` is wired into
`build_order_carbon` yet, and there is no monitor on the `customer_id` ↔ `cust_ref` identity
resolution match rate in `build_customer_360` (nor on unresolvable `cust_ref = 'n/a'` rows in
`campaign_membership`), so a spike in unmatched or unresolvable marketing records would silently
drop from `customer_360` without alarming.

## Common Mistakes to Avoid

- **Conflating the two lineage paths** (e.g., implying `customer` feeds `quarterly_sustainability`) —
  the customer PII pipeline (`customer` → `customer_360` → `marketing_audience`) and the CSRD carbon
  pipeline (`orders` + `shipment` → `order_carbon` → `quarterly_sustainability`) are separate subgraphs.
- **Omitting a PII breach / access anomaly signal** — tracing a `customer` PII breach requires an
  access-log signal mapped to GDPR Art. 33's 72-hour breach notification SLA, not just data-quality checks.
- **Confusing downstream and upstream** — blast radius is downstream; report provenance is upstream.
- **Thresholds with no owner** — a signal nobody owns is not monitoring.
- **No gap analysis** — the strongest answer names what the current signals would miss.